In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "sofie"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

DB_PATH = DATA_DIR / "sofie_data.db"

conn = sqlite3.connect(
    f"file:{DB_PATH.as_posix()}?mode=ro",
    uri=True
)

conn.execute("PRAGMA query_only=ON")

In [2]:
query = """
SELECT
    timestamp,
    P_sum
FROM logging
ORDER BY timestamp
"""

site_meter = pd.read_sql_query(query, conn)

site_meter.head()

,timestamp,P_sum
0,2025-12-01 00:00:02.000000,1
1,2025-12-01 00:00:07.000000,1
2,2025-12-01 00:00:12.000000,1
3,2025-12-01 00:00:17.000000,1
4,2025-12-01 00:00:22.000000,1


In [3]:
site_meter["timestamp"] = pd.to_datetime(site_meter["timestamp"])

print("Dataset info:")
site_meter.info()

print("\nMissing values:")
print(site_meter.isna().sum())

print("\nDuplicate timestamps:")
print(site_meter["timestamp"].duplicated().sum())

print("\nRaw power statistics:")
print(site_meter["P_sum"].describe())

print("\nNegative readings:")
print((site_meter["P_sum"] < 0).sum())

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 3607431 entries, 0 to 3607430
Data columns (total 2 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   P_sum      int64         
dtypes: datetime64[us](1), int64(1)
memory usage: 55.0 MB

Missing values:
timestamp    0
P_sum        0
dtype: int64

Duplicate timestamps:
3

Raw power statistics:
count    3.607431e+06
mean    -2.302393e+01
std      2.254057e+02
min     -2.111000e+03
25%     -2.000000e+01
50%      3.000000e+00
75%      8.000000e+00
max      1.752000e+03
Name: P_sum, dtype: float64

Negative readings:
1535061


In [4]:
# Convert P_sum from its stored scale to kW
site_meter["site_power_kw"] = site_meter["P_sum"] / 100

print("Power statistics in kW:")
print(site_meter["site_power_kw"].describe())

print("\nDuplicate timestamp records:")
duplicates = site_meter[
    site_meter["timestamp"].duplicated(keep=False)
].sort_values("timestamp")

display(duplicates)

Power statistics in kW:
count    3.607431e+06
mean    -2.302393e-01
std      2.254057e+00
min     -2.111000e+01
25%     -2.000000e-01
50%      3.000000e-02
75%      8.000000e-02
max      1.752000e+01
Name: site_power_kw, dtype: float64

Duplicate timestamp records:


,timestamp,P_sum,site_power_kw
369862,2025-12-22 09:52:02,-41,-0.41
369863,2025-12-22 09:52:02,-43,-0.43
369864,2025-12-22 09:52:07,-42,-0.42
369865,2025-12-22 09:52:07,-44,-0.44
369866,2025-12-22 09:52:12,-43,-0.43
369867,2025-12-22 09:52:12,-46,-0.46


In [5]:
# Average readings that share the same timestamp
site_meter_clean = (
    site_meter.groupby("timestamp", as_index=False)
    .agg(site_power_kw=("site_power_kw", "mean"))
)

print("Rows before:", len(site_meter))
print("Rows after:", len(site_meter_clean))

# Resample into 15-minute intervals
site_meter_15min = (
    site_meter_clean
    .set_index("timestamp")
    .resample("15min")
    .agg(
        site_power_kw=("site_power_kw", "mean"),
        reading_count=("site_power_kw", "count")
    )
)

# Identify intervals with and without readings
site_meter_15min["data_available"] = (
    site_meter_15min["reading_count"] > 0
)

print("\n15-minute dataset shape:")
print(site_meter_15min.shape)

print("\nReading count statistics:")
print(site_meter_15min["reading_count"].describe())

print("\nMissing intervals:")
print((~site_meter_15min["data_available"]).sum())

display(site_meter_15min.head())

Rows before: 3607431
Rows after: 3607428

15-minute dataset shape:
(20256, 3)

Reading count statistics:
count    20256.000000
mean       178.091825
std         18.349017
min          0.000000
25%        180.000000
50%        180.000000
75%        180.000000
max        181.000000
Name: reading_count, dtype: float64

Missing intervals:
211


,site_power_kw,reading_count,data_available
timestamp,,,
2025-12-01 00:00:00,0.01,180,True
2025-12-01 00:15:00,0.01,180,True
2025-12-01 00:30:00,0.01,180,True
2025-12-01 00:45:00,0.01,180,True
2025-12-01 01:00:00,0.01,180,True


In [6]:
site_meter_output = site_meter_15min.reset_index()

output_path = OUTPUT_DIR / "site_meter_15min.csv"

site_meter_output.to_csv(output_path, index=False)

print("Exported to:", output_path)
print("File shape:", site_meter_output.shape)

Exported to: c:\Users\leena\OneDrive\Desktop\ev-microgrid-twin\data\processed\site_meter_15min.csv
File shape: (20256, 4)
